# TabPFN vs. the tabular ladder, and with the opposition signal

**Runs in Google Colab on a GPU** (Runtime → Change runtime type → **T4 GPU**). About 15–25 minutes in total.

**The question, in course terms:** TabPFN is a transformer pre-trained on millions of synthetic tables (Lecture 12's pre-training idea, applied to tables). Does it beat the tuned models, against Lecture 8's finding that tree ensembles usually win on tabular data?

**What it runs (validation only; the 2019–2020 test cohort is never scored):**
1. `pipeline`: historical rate, LASSO, gradient boosting and TabPFN. Each model gets **rolling CV** (2010–2014) and **out-of-time** (fit 2005–2014, score 2015–2018) for its selected configuration.
2. `stack`: TabPFN as the first stage, then the Sabin opposition flag as a second stage on 2015–2018. LASSO and gradient boosting run too, for comparison in the same environment.

**One-time setup, under Secrets (🔑) in the left sidebar:**
- `GITHUB_TOKEN`: a GitHub personal access token with read access to the private repo.
- `TABPFN_TOKEN`: from https://ux.priorlabs.ai, after accepting the model license (non-commercial use; fine for a class project).

Blank fields are imputed before TabPFN sees them (no missing-value flags), matching the Nov 15 leakage fix.

**When it finishes:** File → Download → Download .ipynb, then save it to `notebooks/` so the results can be read and logged.

In [1]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> T4 GPU, then rerun"

GPU 0: Tesla T4 (UUID: GPU-0cf42462-e6ac-f89f-8dc7-ad4e56641165)


In [2]:
from google.colab import userdata
import os, subprocess
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("queue-risk-ai"):
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/devault-settings/queue-risk-ai.git"], check=True)
%cd queue-risk-ai
!git pull -q && git log --oneline -1

/content/queue-risk-ai
65d04b8 (HEAD -> main, origin/main, origin/HEAD) Prepare the TabPFN run: impute before TabPFN, add it as a stage-1 option


In [3]:
!pip install -q -e . tabpfn
!apt-get -qq install -y poppler-utils > /dev/null   # pdftotext, for the Sabin reports
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
import tabpfn, torch; print("tabpfn", tabpfn.__version__, "| CUDA:", torch.cuda.is_available())

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for queue_risk (pyproject.toml) ... done
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/p/poppler/poppler-utils_24.02.0-1ubuntu9.10_amd64.deb  404  Not Found [IP: 104.20.28.246 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
tabpfn 10.2.0 | CUDA: True


In [4]:
!python tests/selftest.py | tail -3

  [ok  ] only resolved outcomes are labeled

All checks passed.


## 1. The model ladder with TabPFN

Each line prints the rolling-CV AUC and the out-of-time AUC on 2015–2018 for the **same** selected configuration. The reference numbers from Nov 15 (CPU, no TabPFN) are LASSO **0.631 / 0.686** and gradient boosting **0.663 / 0.633**.

In [5]:
!python -W ignore -m queue_risk.pipeline --models "Historical,LASSO,Gradient,TabPFN"

outcome: built within 5y | train (2005, 2014): 4,029 projects, 11.1% | test (2019, 2020): 3,184 projects, 3.7% | 25 features, 5 rolling folds
  Historical rate (region x tech)                  CV AUC 0.620 | out-of-time 0.593  (0.1s)
  LASSO logistic                                   CV AUC 0.635 | out-of-time 0.682  (51.5s)
  Gradient boosting                                CV AUC 0.666 | out-of-time 0.651  (61.4s)

tabpfn-v3.5-20260909.safetensors: downloading bytes:  36% 319M/876M [00:02<00:02, 217MB/s, 28.0MB/s  ]
tabpfn-v3.5-20260909.safetensors: downloading bytes:  61% 531M/876M [00:02<00:01, 332MB/s, 43.4MB/s  ]
tabpfn-v3.5-20260909.safetensors: downloading bytes:  67% 588M/876M [00:03<00:00, 313MB/s, 48.8MB/s  ]
tabpfn-v3.5-20260909.safetensors: downloading bytes:  94% 819M/876M [00:03<00:00, 281MB/s, 64.9MB/s  ]
tabpfn-v3.5-20260909.safetensors: reconstructing file:  69% 603M/876M [00:04<00:01, 181MB/s, 35.8MB/s  ]
tabpfn-v3.5-20260909.safetensors: reconstructing file:  92% 80

In [6]:
import pandas as pd
pd.read_csv("reports/results_within_validation.csv").drop(columns="secs").round(3)

,model,CV AUC,settings,out-of-time AUC 2015-2018
0,Historical rate (region x tech),0.620,{},0.593
1,LASSO logistic,0.635,{'C': 0.01},0.682
2,Gradient boosting,0.666,"{'learning_rate': 0.03, 'max_depth': 4, 'max_i...",0.651
3,TabPFN (pre-trained transformer),0.677,{},0.646


## 2. TabPFN as the first stage, plus the opposition flag

The first command downloads the five Sabin editions from the Internet Archive and parses them (a few minutes). The stacking check then fits the second stage leave-one-year-out on 2015–2018.

In [7]:
!python -W ignore -m queue_risk.sabin | tail -3

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/queue-risk-ai/src/queue_risk/sabin.py", line 311, in <module>
    main()
    ~~~~^^
  File "/content/queue-risk-ai/src/queue_risk/sabin.py", line 237, in main
    r, c = build(raw)
           ~~~~~^^^^^
  File "/content/queue-risk-ai/src/queue_risk/sabin.py", line 202, in build
    r, c = parse_edition(latest)
           ~~~~~~~~~~~~~^^^^^^^^
  File "/content/queue-risk-ai/src/queue_risk/sabin.py", line 154, in parse_edition
    secs = state_sections(edition_text(edition))
                          ~~~~~~~~~~~~^^^^^^^^^
  File "/content/queue-risk-ai/src/queue_risk/sabin.py", line 80, in edition_text
    subprocess.run(["pdftotext", "-layout", str(pdf), str(txt)], check=True, capture_output=True)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.

In [8]:
!python -W ignore -m queue_risk.stack --stage1 tabpfn

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/queue-risk-ai/src/queue_risk/stack.py", line 109, in <module>
    main()
    ~~~~^^
  File "/content/queue-risk-ai/src/queue_risk/stack.py", line 74, in main
    c = pd.read_csv(config.INTERIM / "sabin_contested.csv", dtype={"fips_list": str})
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1880, in

In [9]:
!python -W ignore -m queue_risk.stack --stage1 both    # LASSO and gradient boosting, same environment, for comparison

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/queue-risk-ai/src/queue_risk/stack.py", line 109, in <module>
    main()
    ~~~~^^
  File "/content/queue-risk-ai/src/queue_risk/stack.py", line 74, in main
    c = pd.read_csv(config.INTERIM / "sabin_contested.csv", dtype={"fips_list": str})
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py", line 1880, in

Done. **File → Download → Download .ipynb** and save it as `notebooks/tabpfn_colab.ipynb`, replacing this copy, so the outputs can be read and logged in `docs/model_log.md`.

In [10]:
import os
from google.colab import userdata
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")   # in case the session restarted

!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils > /dev/null
!which pdftotext
%cd /content/queue-risk-ai
!git pull -q
!rm -f data/external/sabin/*.txt
!python -W ignore -m queue_risk.sabin | tail -3
!python -W ignore -m queue_risk.stack --stage1 tabpfn
!python -W ignore -m queue_risk.stack --stage1 both

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/usr/bin/pdftotext
/content/queue-risk-ai
Earliest year in contested text: {2000.0: 3, 2002.0: 1, 2003.0: 1, 2004.0: 1, 2005.0: 8, 2006.0: 3, 2007.0: 4, 2008.0: 4, 2009.0: 4, 2010.0: 17, 2011.0: 17, 2012.0: 15, 2013.0: 17, 2014.0: 20, 2015.0: 28, 2016.0: 36, 2017.0: 23, 2018.0: 41, 2019.0: 37, 2020.0: 32, 2021.0: 56, 2022.0: 65, 2023.0: 67, 2024.0: 45, 2025.0: 24, 2026.0: 1}
Rule-based outcome on all 572 paragraphs: {'blocked': 261, 'proceeded': 156, 'unresolved': 155}
Labeling sheet: data/interim/sabin_label_sheet.csv (see docs/labeling_guidelines.md)

=== stage 1: TabPFN (pre-trained transformer) (pipeline-selected settings)
Validation, cohorts 2015-2018 (3,490 projects, 13% in a county with earlier contested projects)
  stage-1 model alone, AUC 0.646
  stage 2 out-of-fold AUC: tabular only 0.635 |